# Feature Engineering — Product-City Demand Features

Reads the Gold layer (`product_city_demand`) and engineers time-series features for demand forecasting:
lag features, rolling windows, and calendar features. The resulting feature
table is published to the Databricks Feature Store as a **time series feature table** for point-in-time
correct training set construction. Uses an OOP class-based design (`FeatureEngineering`) with Python
`logging` for structured output.

**Pipeline:**

```
Gold: product_city_demand  →  Feature Engineering  →  Feature Store: product_city_demand_features
```

**Engineered features:**

| Category | Features |
|---|---|
| Lag | `lag_1_demand`, `lag_7_demand` |
| Rolling (7-row) | `rolling_7_avg_demand`, `rolling_7_sum_demand`, `rolling_7_max_demand` |
| Calendar | `day_of_week`, `day_of_month`, `is_weekend`, `quarter` |

**Feature Store:** Time series table with primary keys `(product_name, destination_city, transaction_date)` and timeseries column `transaction_date`.

In [0]:
%pip install databricks-feature-engineering

dbutils.library.restartPython()

In [0]:
import yaml
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.feature_engineering import FeatureEngineeringClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


class BaseLayer:
    """Base class for medallion layer ingestion with shared config and logging."""

    def __init__(self, config_path: str, layer_name: str):
        self.config_path = config_path
        self.layer_name = layer_name
        self.logger = logging.getLogger(layer_name)
        self._load_config()

    def _load_config(self) -> None:
        """Load YAML configuration from the configured path."""
        with open(self.config_path, "r") as f:
            self.cfg = yaml.safe_load(f)
        self.catalog = self.cfg["catalog"]
        self.logger.info(f"Config loaded from {self.config_path}")

    def ensure_schema(self, schema_name: str) -> None:
        """Create the target schema in Unity Catalog if it doesn't exist."""
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{schema_name}")
        self.logger.info(f"Ensured schema exists: {self.catalog}.{schema_name}")


class FeatureEngineering(BaseLayer):
    """Feature Engineering layer: transform Gold data into ML-ready features for the Feature Store."""

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "feature_engineering")
        self.layer_cfg = self.cfg["feature_engineering"]
        self.gold_schema = self.cfg["schemas"]["gold"]
        self.features_schema = self.cfg["schemas"]["features"]
        self.gold_table = self.layer_cfg["source"]["gold_table"]
        self.feature_table = self.layer_cfg["target"]["feature_table"]
        self.gold_full = f"{self.catalog}.{self.gold_schema}.{self.gold_table}"
        self.feature_full = f"{self.catalog}.{self.features_schema}.{self.feature_table}"
        self.primary_keys = self.layer_cfg["primary_keys"]
        self.timeseries_column = self.layer_cfg["timeseries_column"]
        self.fe = FeatureEngineeringClient()
        self.logger.info(f"Gold source:      {self.gold_full}")
        self.logger.info(f"Feature target:   {self.feature_full}")

    # ------------------------------------------------------------------
    # Read
    # ------------------------------------------------------------------
    def read_gold(self):
        """Read the Gold Delta table (aggregated product-city-date demand)."""
        self.gold_df = spark.table(self.gold_full)
        self.gold_count = self.gold_df.count()
        self.logger.info(f"Gold rows read: {self.gold_count:,}")
        self.logger.info(f"Gold columns:   {len(self.gold_df.columns)}")
        return self.gold_df

    # ------------------------------------------------------------------
    # Calendar / time features
    # ------------------------------------------------------------------
    def create_time_features(self, df):
        """Add calendar/time features derived from transaction_date."""
        self.logger.info("Adding calendar/time features...")
        df = (
            df
            .withColumn("day_of_week", F.dayofweek("transaction_date"))
            .withColumn("day_of_month", F.dayofmonth("transaction_date"))
            .withColumn("is_weekend", F.dayofweek("transaction_date").isin(1, 7))
            .withColumn("quarter", F.quarter("transaction_date"))
        )
        self.logger.info(f"Columns after time features: {len(df.columns)}")
        return df

    # ------------------------------------------------------------------
    # Lag features
    # ------------------------------------------------------------------
    def create_lag_features(self, df):
        """Add lag features partitioned by (product_name, destination_city), ordered by transaction_date."""
        self.logger.info("Adding lag features...")
        lag_windows = self.layer_cfg["features"]["lag_windows"]
        lag_columns = self.layer_cfg["features"]["lag_columns"]
        w = Window.partitionBy("product_name", "destination_city").orderBy("transaction_date")

        for col_name in lag_columns:
            for lag_n in lag_windows:
                if col_name == "demand_quantity":
                    feat_name = f"lag_{lag_n}_demand"
                elif col_name == "ordered_quantity":
                    feat_name = f"lag_{lag_n}_ordered_quantity"
                else:
                    feat_name = f"lag_{lag_n}_{col_name}"
                df = df.withColumn(feat_name, F.lag(col_name, lag_n).over(w))
                self.logger.info(f"  Created {feat_name}")

        self.logger.info(f"Columns after lag features: {len(df.columns)}")
        return df

    # ------------------------------------------------------------------
    # Rolling window features
    # ------------------------------------------------------------------
    def create_rolling_features(self, df):
        """Add rolling window features (7-row avg/sum/max/min) partitioned by product-city."""
        self.logger.info("Adding rolling window features...")
        rolling_window = self.layer_cfg["features"]["rolling_window"]
        rolling_columns = self.layer_cfg["features"]["rolling_columns"]
        w = (
            Window
            .partitionBy("product_name", "destination_city")
            .orderBy("transaction_date")
            .rowsBetween(-(rolling_window - 1), 0)
        )

        for col_name in rolling_columns:
            if col_name == "demand_quantity":
                df = (
                    df
                    .withColumn(f"rolling_{rolling_window}_avg_demand", F.avg(col_name).over(w))
                    .withColumn(f"rolling_{rolling_window}_sum_demand", F.sum(col_name).over(w))
                    .withColumn(f"rolling_{rolling_window}_max_demand", F.max(col_name).over(w))
                )
                self.logger.info(f"  Created rolling_{rolling_window}_avg/sum/max_demand")
            else:
                df = df.withColumn(f"rolling_{rolling_window}_avg_{col_name}", F.avg(col_name).over(w))
                self.logger.info(f"  Created rolling_{rolling_window}_avg_{col_name}")

        self.logger.info(f"Columns after rolling features: {len(df.columns)}")
        return df

    # ------------------------------------------------------------------
    # Assemble final DataFrame
    # ------------------------------------------------------------------
    def assemble(self):
        """Assemble the final feature DataFrame with a deterministic column order."""
        self.logger.info("Assembling final feature DataFrame...")

        pk_cols = ["product_name", "destination_city", "transaction_date"]
        original_cols = [
            "transaction_month", "product_category", "quantity_unit",
            "demand_quantity",
            "avg_unit_price_usd", "avg_available_inventory",
        ]
        engineered_cols = [
            "lag_1_demand", "lag_7_demand",
            "rolling_7_avg_demand", "rolling_7_sum_demand",
            "rolling_7_max_demand",
            "day_of_week", "day_of_month", "is_weekend", "quarter",
        ]
        final_cols = pk_cols + original_cols + engineered_cols
        self.feature_df = self.feature_df.select(*final_cols)
        self.feature_count = self.feature_df.count()
        self.logger.info(f"Final feature rows:    {self.feature_count:,}")
        self.logger.info(f"Final feature columns: {len(self.feature_df.columns)}")
        return self.feature_df

    # ------------------------------------------------------------------
    # Write to Feature Store
    # ------------------------------------------------------------------
    def write_feature_store(self) -> None:
        """Write the feature DataFrame to the Databricks Feature Store as a time series feature table."""
        self.logger.info("Writing to Feature Store...")
        self.ensure_schema(self.features_schema)

        # Check if the feature table already exists
        table_exists = spark.catalog.tableExists(self.feature_full)

        if table_exists:
            self.logger.info(f"Feature table already exists — dropping and recreating...")
            spark.sql(f"DROP TABLE IF EXISTS {self.feature_full}")

        self.fe.create_table(
            name=self.feature_full,
            primary_keys=self.primary_keys,
            timeseries_column=self.timeseries_column,
            df=self.feature_df,
            schema=self.feature_df.schema,
            description=(
                "Product-city demand features for forecasting: lag, rolling window, "
                "calendar, and derived change features. Time series feature table "
                "keyed by (product_name, destination_city, transaction_date)."
            ),
        )
        self.logger.info(f"Created feature table: {self.feature_full}")
        self.logger.info(f"  Primary keys:       {', '.join(self.primary_keys)}")
        self.logger.info(f"  Timeseries column:  {self.timeseries_column}")
        self.logger.info(f"  Rows written:       {self.feature_count:,}")

    # ------------------------------------------------------------------
    # Validate
    # ------------------------------------------------------------------
    def validate(self) -> None:
        """Display sample records, schema, null counts, and final summary."""
        feature_written = spark.table(self.feature_full)
        self.logger.info("=" * 60)
        self.logger.info("FEATURE ENGINEERING SUMMARY")
        self.logger.info("=" * 60)
        self.logger.info(f"  Gold rows (source):         {self.gold_count:,}")
        self.logger.info(f"  Feature rows (engineered):  {self.feature_count:,}")
        self.logger.info(f"  Feature columns:            {len(feature_written.columns)}")
        self.logger.info(f"  Feature table:              {self.feature_full}")
        self.logger.info(f"  Primary keys:               {', '.join(self.primary_keys)}")
        self.logger.info(f"  Timeseries column:          {self.timeseries_column}")
        self.logger.info("=" * 60)

        self.logger.info("Feature table sample (10 rows):")
        display(feature_written.limit(10))

        self.logger.info("Feature table schema:")
        feature_written.printSchema()

        self.logger.info("Null counts per engineered feature column:")
        engineered_cols = [
            "lag_1_demand", "lag_7_demand",
            "rolling_7_avg_demand", "rolling_7_sum_demand",
            "rolling_7_max_demand", "day_of_week", "day_of_month",
            "is_weekend", "quarter",
        ]
        null_counts = feature_written.select(
            [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in engineered_cols]
        ).collect()[0]
        for c in engineered_cols:
            self.logger.info(f"  {c:30s} nulls: {null_counts[c]:,}")
        self.logger.info("=" * 60)

    # ------------------------------------------------------------------
    # Orchestrate
    # ------------------------------------------------------------------
    def run(self) -> None:
        """Execute the full feature engineering pipeline."""
        self.read_gold()
        self.feature_df = self.gold_df
        self.feature_df = self.create_time_features(self.feature_df)
        self.feature_df = self.create_lag_features(self.feature_df)
        self.feature_df = self.create_rolling_features(self.feature_df)
        self.assemble()
        self.write_feature_store()
        self.validate()

In [0]:
# Execute feature engineering pipeline
fe_engineer = FeatureEngineering()
fe_engineer.run()